# LVLM Training: Debugging & Troubleshooting Guide

This notebook provides diagnostic tools and common solutions for training issues.

**Common Issues Covered:**
1. Data loading errors
2. CUDA out of memory
3. Training instability
4. Checkpoint corruption
5. Evaluation metrics anomalies

In [ ]:
# Import libraries
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import sys
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("📋 ENVIRONMENT DIAGNOSTICS")
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"GPU utilized: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB / {torch.cuda.max_memory_allocated(0) / 1e9:.2f} GB (peak)")

## 1. Data Loading Diagnostics

In [ ]:
import os

print("\n📂 CHECKING DATA STRUCTURE")
print("="*60)

# Expected directory structure
required_dirs = [
    '../data/tvqa',
    '../data/tvqa/frames_frcnn',  # Feature frames
    '../data/tvqa/metadata',       # Annotations
    '../data/activitynet-qa',
    '../data/activitynet-qa/frames',
    '../data/activitynet-qa/metadata',
]

required_files = {
    '../data/tvqa/metadata/tvqa_train.json': 'TVQA training set',
    '../data/tvqa/metadata/tvqa_val.json': 'TVQA validation set',
    '../data/tvqa/metadata/tvqa_test.json': 'TVQA test set',
    '../data/activitynet-qa/metadata/train.json': 'ActivityNet-QA training set',
    '../data/activitynet-qa/metadata/val.json': 'ActivityNet-QA validation set',
}

def check_paths(dirs, files):
    status = {'missing_dirs': [], 'found_dirs': [], 'missing_files': [], 'found_files': []}
    
    for d in dirs:
        if Path(d).exists():
            status['found_dirs'].append(d)
        else:
            status['missing_dirs'].append(d)
    
    for f, description in files.items():
        if Path(f).exists():
            status['found_files'].append((f, description))
        else:
            status['missing_files'].append((f, description))
    
    return status

status = check_paths(required_dirs, required_files)

print(f"✓ Directories found: {len(status['found_dirs'])}")
for d in status['found_dirs']:
    print(f"    {d}")

if status['missing_dirs']:
    print(f"\n✗ Directories missing: {len(status['missing_dirs'])}")
    for d in status['missing_dirs']:
        print(f"    {d}")

print(f"\n✓ Files found: {len(status['found_files'])}")
for f, desc in status['found_files']:
    file_size = Path(f).stat().st_size / 1e6
    print(f"    {desc}: {file_size:.1f} MB")

if status['missing_files']:
    print(f"\n✗ Files missing: {len(status['missing_files'])}")
    for f, desc in status['missing_files']:
        print(f"    {desc}: {f}")

print(f"\n📊 DATA STATUS: {'✓ READY' if not status['missing_files'] else '⚠ INCOMPLETE'}")

## 2. Memory Usage Analysis

In [ ]:
print("\n💾 GPU MEMORY ANALYSIS")
print("="*60)

if torch.cuda.is_available():
    # Current memory usage
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"Total GPU memory: {total:.1f} GB")
    print(f"Currently allocated: {allocated:.2f} GB ({allocated/total*100:.1f}%)")
    print(f"Currently reserved: {reserved:.2f} GB ({reserved/total*100:.1f}%)")
    print(f"Available: {total - reserved:.2f} GB")
    
    # Estimate batch sizes for different configurations
    print("\n📋 RECOMMENDED BATCH SIZES:")
    configs = [
        ('FP32, K=3', 0.5, 16),
        ('FP32, K=3', 1.0, 8),
        ('FP16, K=3', 0.5, 32),
        ('FP16, K=3', 1.0, 24),
        ('FP16, K=5', 1.0, 16),
    ]
    
    for config, memory_per_sample, batch_size in configs:
        required = memory_per_sample * batch_size
        status = '✓' if required < total * 0.8 else '✗'
        print(f"    {status} {config:20s} @ batch_size={batch_size:2d}: {required:.2f} GB")
else:
    print("⚠ CUDA not available. Training will use CPU (very slow).")
    print("  Recommended: Use GPU-enabled environment")

## 3. Training Stability Diagnostics

In [ ]:
# Generate synthetic training logs to simulate different scenarios
def generate_training_logs(scenario='healthy'):
    """Generate synthetic training logs for different scenarios."""
    epochs = 20
    
    if scenario == 'healthy':
        # Healthy training: smooth loss decrease
        train_loss = 2.5 * np.exp(-np.arange(epochs) / 5) + 0.3 + np.random.normal(0, 0.05, epochs)
        val_loss = 2.6 * np.exp(-np.arange(epochs) / 5) + 0.35 + np.random.normal(0, 0.06, epochs)
        accuracy = 0.35 * (1 - np.exp(-np.arange(epochs) / 3)) + np.random.normal(0, 0.02, epochs)
    
    elif scenario == 'diverging':
        # Diverging: loss explodes (learning rate too high)
        train_loss = 0.8 + np.arange(epochs) * 0.15 + np.random.normal(0, 0.1, epochs)
        val_loss = 0.9 + np.arange(epochs) * 0.18 + np.random.normal(0, 0.15, epochs)
        accuracy = np.random.normal(0.5, 0.01, epochs)
    
    elif scenario == 'plateaued':
        # Plateaued: loss stuck, no improvement
        train_loss = 0.5 + 0.05 * np.sin(np.arange(epochs)) + np.random.normal(0, 0.03, epochs)
        val_loss = 0.55 + 0.06 * np.sin(np.arange(epochs)) + np.random.normal(0, 0.04, epochs)
        accuracy = 0.62 * np.ones(epochs) + np.random.normal(0, 0.01, epochs)
    
    elif scenario == 'overfitting':
        # Overfitting: train loss decreases but val loss increases
        train_loss = 0.3 * np.exp(-np.arange(epochs) / 3) + np.random.normal(0, 0.02, epochs)
        val_loss = 0.5 * np.exp(-np.arange(epochs) / 8) + 0.3 * np.arange(epochs) / epochs + np.random.normal(0, 0.05, epochs)
        accuracy = 0.55 * (1 - np.exp(-np.arange(epochs) / 2)) + np.random.normal(0, 0.02, epochs)
    
    return train_loss, val_loss, accuracy

scenarios = ['healthy', 'diverging', 'plateaued', 'overfitting']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training Stability Diagnostics', fontsize=14, fontweight='bold')

for idx, scenario in enumerate(scenarios):
    ax = axes[idx // 2, idx % 2]
    train_loss, val_loss, accuracy = generate_training_logs(scenario)
    
    epochs = len(train_loss)
    ax.plot(train_loss, label='Train Loss', marker='o', markersize=4, alpha=0.7)
    ax.plot(val_loss, label='Val Loss', marker='s', markersize=4, alpha=0.7)
    ax.fill_between(range(epochs), train_loss, val_loss, alpha=0.2)
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(f'Scenario: {scenario.upper()}', fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 TRAINING SCENARIO INTERPRETATION:")
print("="*60)
print("\n1. HEALTHY:")
print("   ✓ Both losses decrease smoothly")
print("   ✓ Val loss slightly higher than train (normal)")
print("   ✓ Training is progressing well")

print("\n2. DIVERGING:")
print("   ✗ Losses rapidly increase")
print("   ✗ Model is unstable")
print("   → Solution: Reduce learning rate by 10x, check gradient clipping")

print("\n3. PLATEAUED:")
print("   ⚠ Losses stuck, no improvement")
print("   ⚠ May have reached local minimum or learning rate too small")
print("   → Solution: Increase learning rate, check data augmentation")

print("\n4. OVERFITTING:")
print("   ⚠ Train loss decreases but val loss increases")
print("   ⚠ Model memorizing training data")
print("   → Solution: Add dropout/regularization, reduce model size, augment data")

## 4. Checkpoint Validation

In [ ]:
print("\n🔐 CHECKPOINT VALIDATION SYSTEM")
print("="*60)

checkpoint_dir = Path('../experiments/checkpoints')
print(f"\nCheckpoint directory: {checkpoint_dir}")
print(f"Directory exists: {checkpoint_dir.exists()}")

if checkpoint_dir.exists():
    checkpoint_files = list(checkpoint_dir.glob('*.pt'))
    print(f"\nFound {len(checkpoint_files)} checkpoint files:")
    
    for ckpt in sorted(checkpoint_files):
        try:
            file_size = ckpt.stat().st_size / 1e6
            # Try to load metadata
            ckpt_data = torch.load(ckpt, map_location='cpu')
            
            if isinstance(ckpt_data, dict):
                keys = list(ckpt_data.keys())
                epoch = ckpt_data.get('epoch', 'N/A')
                step = ckpt_data.get('global_step', 'N/A')
                status = "✓ VALID"
            else:
                status = "⚠ Unknown format"
                keys = []
        except Exception as e:
            status = f"✗ CORRUPTED: {str(e)[:30]}"
            epoch, step, keys = 'N/A', 'N/A', []
        
        print(f"  {status:20s} {ckpt.name:30s} {file_size:8.1f} MB (Epoch {epoch}, Step {step})")
else:
    print("  ⚠ No checkpoints found. Run training first.")

print("\n✅ CHECKPOINT REQUIREMENTS:")
print("  • model_state_dict: Model weights")
print("  • optimizer_state_dict: Optimizer state")
print("  • scheduler_state_dict: Learning rate scheduler state")
print("  • epoch: Current epoch number")
print("  • global_step: Total training steps")
print("  • best_accuracy: Best validation accuracy so far")

## 5. Common Issues & Solutions

In [ ]:
issues_solutions = pd.DataFrame([
    {
        'Problem': 'CUDA out of memory',
        'Symptoms': 'RuntimeError: CUDA out of memory',
        'Solutions': [
            'Reduce batch_size in config (32 → 16)',
            'Reduce max_depth from 5 to 3',
            'Enable gradient accumulation steps in config',
            'Use FP16 mixed precision (already enabled)',
        ]
    },
    {
        'Problem': 'Data loading errors',
        'Symptoms': 'FileNotFoundError: Annotations not found',
        'Solutions': [
            'Check data directory structure (diagnostic above)',
            'Download datasets: tvqa, activitynet-qa',
            'Verify JSON annotation files are readable',
            'Use mock mode for testing: --mock flag',
        ]
    },
    {
        'Problem': 'NaN loss during training',
        'Symptoms': 'Loss becomes NaN or inf',
        'Solutions': [
            'Reduce learning rate (1e-4 → 5e-5)',
            'Enable gradient clipping (already at 1.0)',
            'Check for zero-division in loss computation',
            'Verify data normalization is correct',
        ]
    },
    {
        'Problem': 'Slow training',
        'Symptoms': 'Training takes hours per epoch',
        'Solutions': [
            'Cache features to HDF5: python extract_features.py',
            'Increase batch_size (if memory allows)',
            'Use num_workers > 0 for data loading',
            'Verify GPU is being used (nvidia-smi)',
        ]
    },
    {
        'Problem': 'Poor model accuracy',
        'Symptoms': 'Accuracy stuck at 45-50%',
        'Solutions': [
            'Train for more epochs (20 → 50)',
            'Verify data loading (check dataset statistics)',
            'Try different learning rates (1e-4 to 1e-3)',
            'Check temporal binding is working in forward()',
        ]
    },
    {
        'Problem': 'Checkpoint loading error',
        'Symptoms': 'RuntimeError: Error(s) loading state_dict',
        'Solutions': [
            'Verify model architecture matches checkpoint',
            'Use map_location for CPU-GPU compatibility',
            'Check PyTorch version compatibility',
            'Delete corrupted checkpoint and retrain',
        ]
    },
], columns=['Problem', 'Symptoms', 'Solutions'])

print("\n⚠️ COMMON ISSUES & TROUBLESHOOTING")
print("="*60)

for idx, row in issues_solutions.iterrows():
    print(f"\n{idx+1}. {row['Problem'].upper()}")
    print(f"   Symptoms: {row['Symptoms']}")
    print(f"   Solutions:")
    for solution in row['Solutions']:
        print(f"      • {solution}")

## 6. Quick Health Check

In [ ]:
print("\n🏥 SYSTEM HEALTH CHECK")
print("="*60)

checks = {
    'Python 3.8+': sys.version_info >= (3, 8),
    'PyTorch installed': torch is not None,
    'CUDA available': torch.cuda.is_available(),
    'GPU with enough memory': torch.cuda.is_available() and (torch.cuda.get_device_properties(0).total_memory / 1e9) >= 6,
    'Data locally available': Path('../data/tvqa').exists() or Path('../data/activitynet-qa').exists(),
    'Checkpoint directory exists': checkpoint_dir.exists(),
}

status_count = sum(checks.values())
total_count = len(checks)

for check, status in checks.items():
    symbol = '✓' if status else '✗'
    print(f"{symbol} {check}")

print(f"\n📊 Overall Health: {status_count}/{total_count} checks passed")

if status_count == total_count:
    print("✅ System is ready for training!")
elif status_count >= 5:
    print("⚠️ System is mostly ready. See above for missing components.")
else:
    print("❌ System needs setup before training can begin.")

print("\n💡 NEXT STEPS:")
print("  1. Run: python extract_features.py --dataset tvqa")
print("  2. Run: python train.py --dataset tvqa --epochs 20")
print("  3. Monitor training with: tail -f experiments/logs/training.log")
print("  4. View results with notebooks: 03_ablation_analysis.ipynb")